In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras import backend as K
import numpy as np
import cv2


def euclidean_distance(vects):
    x, y = vects
    sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sum_square, K.epsilon()))

def contrastive_loss(y_true, y_pred):
    margin = 1.0
    y_true = tf.cast(y_true, y_pred.dtype)
    square_pred = K.square(y_pred)
    margin_square = K.square(K.maximum(margin - y_pred, 0))
    return K.mean(y_true * square_pred + (1 - y_true) * margin_square)

# ✅ FIX: Squeeze y_pred from (batch,1) to (batch,) to match training metric
def contrastive_accuracy(y_true, y_pred):
    y_pred_flat = K.squeeze(y_pred, axis=-1)
    return K.mean(tf.equal(y_true, tf.cast(y_pred_flat < 0.5, y_true.dtype)))

# Load the model
model = load_model('siamese_model_contrastive.h5', custom_objects={
    'contrastive_loss': contrastive_loss,
    'contrastive_accuracy': contrastive_accuracy,
    'euclidean_distance': euclidean_distance
})

print("Model loaded!")

In [ ]:
import matplotlib.pyplot as plt

def preprocess_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # ✅ FIX: Keep original image for display (not the tiny 28x28 version)
    img_display = img

    # ✅ FIX: Better resize to match MNIST style
    img_processed = cv2.resize(img, (28, 28), interpolation=cv2.INTER_AREA)

    # ✅ FIX: Equalize contrast so digit stands out clearly like MNIST
    img_processed = cv2.equalizeHist(img_processed)

    # ✅ FIX: Threshold to pure black/white like MNIST digits
    _, img_processed = cv2.threshold(img_processed, 127, 255, cv2.THRESH_BINARY)

    img_processed = img_processed.astype("float32") / 255.0

    # ✅ FIX: Model input shape is (28,28) — only add batch dimension
    img_processed = np.expand_dims(img_processed, axis=0)  # (1, 28, 28)
    return img_processed, img_display


img1_path = "/content/img1.jpg"
img2_path = "/content/img2.jpg"

img1_processed, img1_display = preprocess_image(img1_path)
img2_processed, img2_display = preprocess_image(img2_path)


distance = model.predict([img1_processed, img2_processed])[0][0]
similarity_status = "SIMILAR" if distance < 0.5 else "DIFFERENT"


plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img1_display, cmap='gray')
plt.title(f"Image 1\nDistance: {distance:.4f}", fontsize=12)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(img2_display, cmap='gray')
plt.title(f"Image 2\nStatus: {similarity_status}", fontsize=12)
plt.axis('off')

plt.suptitle(f"Siamese Network Prediction: Images are {similarity_status}",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Euclidean distance: {distance:.4f}")
print(f"Images are {'SIMILAR' if distance < 0.5 else 'DIFFERENT'} (threshold=0.5)")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- 1. GENERATE SYNTHETIC IMAGES ---
def create_digit_image(digit_type):
    img = np.zeros((28, 28), dtype=np.uint8)

    if digit_type == 7:
        cv2.line(img, (5, 5), (22, 5), 255, thickness=4)
        cv2.line(img, (22, 5), (10, 22), 255, thickness=4)

    elif digit_type == 1:
        cv2.line(img, (14, 5), (14, 23), 255, thickness=4)

    return img

# Create a "7" and a "1"
img_A = create_digit_image(7)
img_B = create_digit_image(1)

# --- 2. PREPARE FOR MODEL ---
img_A_processed = img_A.astype("float32") / 255.0
img_B_processed = img_B.astype("float32") / 255.0

# ✅ FIX: Only add batch dimension — model input is (28,28)
img_A_batch = np.expand_dims(img_A_processed, axis=0)  # (1, 28, 28)
img_B_batch = np.expand_dims(img_B_processed, axis=0)  # (1, 28, 28)

# --- 3. VISUALIZE INPUTS ---
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("Generated '7'")
plt.imshow(img_A, cmap='gray')
plt.subplot(1, 2, 2)
plt.title("Generated '1'")
plt.imshow(img_B, cmap='gray')
plt.show()

# --- 4. PREDICT ---
distance = model.predict([img_A_batch, img_B_batch])[0][0]

print(f"\n------------------------------------------------")
print(f"Calculated Distance: {distance:.4f}")
print(f"------------------------------------------------")

if distance > 0.5:
    print("✅ Result: DIFFERENT (Correct!)")
else:
    print("❌ Result: SIMILAR (Wrong - Model needs more training)")